<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to do fine-tuning of the model and show how it works.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

# Odds Ratio Preference Optimization (ORPO)

In [1]:
# Odds Ratio Preference Optimization (ORPO) - Standalone Implementation
# Innovation: Combines SFT and preference learning in single step
# Loss Function: L_ORPO = L_SFT + λ * L_OR

import warnings

warnings.filterwarnings("ignore")

import subprocess
import sys
import torch
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import json
from datetime import datetime

# ORPO Configuration
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPERATURE = 0.8
MAX_LENGTH = 512
MAX_NEW_TOKENS = 200
LEARNING_RATE_ORPO = 3e-5
NUM_EPOCHS_ORPO = 18
BATCH_SIZE_ORPO = 1
GRAD_ACCUM_ORPO = 12
WARMUP_RATIO = 0.2
LOGGING_STEPS = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

TEST_QUESTIONS = [
    "How do I cook pasta perfectly?",
    "What's the best way to scramble eggs?",
    "How do I make rice that isn't sticky?",
    "What's an easy dinner for beginners?",
    "How do I know when chicken is cooked?",
]


def install_packages():
    packages = [
        "torch>=2.0.0",
        "transformers>=4.36.0",
        "trl>=0.7.4",
        "datasets>=2.14.0",
        "accelerate>=0.21.0",
    ]
    for pkg in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        except:
            pass


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def monitor_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(device) / (1024**3)
        reserved = torch.cuda.memory_reserved(device) / (1024**3)
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
        return reserved < 7.5
    return True


def test_model(model, tokenizer, prompt):
    model.eval()
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    )
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            use_cache=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response.strip()


def evaluate_stage(model, tokenizer, stage_name):
    results = {}
    print(f"\n{'='*60}")
    print(f"{stage_name.upper()} MODEL EVALUATION")
    print(f"{'='*60}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\nQ{i}: {question}")
        print("-" * 40)
        response = test_model(model, tokenizer, question)
        results[question] = response
        print(f"Answer: {response}")

    return results


def create_orpo_dataset():
    """Create comprehensive ORPO preference pairs with strong contrasts"""
    examples = [
        {
            "prompt": "How do I make a simple tomato sauce?",
            "chosen": "Homemade tomato sauce is surprisingly simple and far superior to store-bought versions.\n\nHeat 1/4 cup olive oil in large pan over medium heat. Add 1 diced onion and cook until translucent, about 5 minutes. Add 3 minced garlic cloves and cook 30 seconds until fragrant.\n\nPour in 1 can (28 oz) crushed tomatoes, add 1 tsp salt, 1/2 tsp pepper, and 1 tsp dried basil. Bring to gentle boil, then reduce heat to low.\n\nSimmer uncovered 20-30 minutes, stirring occasionally, until sauce thickens to your liking. Taste and adjust seasoning. Add fresh basil in final 5 minutes if available.",
            "rejected": "Just heat up some tomatoes with garlic and onion until it looks like sauce.",
        },
        {
            "prompt": "What's the best way to cook steak?",
            "chosen": "Perfect steak requires high heat, proper seasoning, and precise timing.\n\nChoose steaks at least 1 inch thick and let them reach room temperature for 30 minutes. Season generously with coarse salt and black pepper.\n\nHeat cast iron pan or grill to high heat. Add small amount of high-smoke-point oil (not olive oil). Sear steak 3-4 minutes per side for medium-rare, depending on thickness.\n\nUse instant-read thermometer: 130-135°F for medium-rare. Let steak rest 5-10 minutes before slicing to redistribute juices.",
            "rejected": "Cook steak on high heat until it looks done. Season with salt and pepper.",
        },
        {
            "prompt": "How do I roast vegetables properly?",
            "chosen": "Proper roasting brings out vegetables' natural sweetness through caramelization.\n\nPreheat oven to 425°F. Cut vegetables into uniform 1-inch pieces for even cooking. Toss with olive oil (1-2 tbsp per baking sheet), salt, and pepper.\n\nSpread in single layer on baking sheet - overcrowding creates steam instead of roasting. Most vegetables take 20-30 minutes.\n\nVegetables are done when fork-tender with golden-brown edges. This browning is crucial for flavor development. Finish with fresh herbs or lemon juice while hot.",
            "rejected": "Put vegetables in the oven with oil until they're soft.",
        },
        {
            "prompt": "How do I make perfect mashed potatoes?",
            "chosen": "Creamy mashed potatoes require the right potato variety and proper technique.\n\nUse starchy potatoes like Russets or Yukon Gold. Peel and cut into uniform chunks. Start in cold, salted water and bring to boil - this ensures even cooking.\n\nCook until fork-tender, about 15-20 minutes. Drain thoroughly and let sit 2-3 minutes to evaporate excess moisture.\n\nMash while hot using potato masher or ricer. Gradually add warm butter and milk/cream, starting with 4 tbsp butter and 1/2 cup liquid per 2 lbs potatoes. Season with salt and white pepper.",
            "rejected": "Boil potatoes and mash them with butter and milk.",
        },
        {
            "prompt": "What's the secret to good fried rice?",
            "chosen": "Great fried rice uses day-old rice and high heat for proper texture and flavor.\n\nUse cold, day-old rice - fresh rice is too moist and creates mushy results. Break up any clumps with your hands before cooking.\n\nHeat wok or large pan over high heat. Add oil and scrambled eggs first, remove and set aside. Add aromatics (garlic, ginger) for 30 seconds.\n\nAdd cold rice, breaking up clumps as you stir-fry for 3-4 minutes. Add soy sauce, vegetables, and cooked eggs back in. Keep everything moving over high heat for best texture.",
            "rejected": "Fry rice with soy sauce and whatever vegetables you have.",
        },
        {
            "prompt": "How do I make bread rise properly?",
            "chosen": "Proper bread rising depends on yeast activation, temperature control, and timing.\n\nProof yeast in warm water (105-110°F) with a pinch of sugar for 5-10 minutes until foamy. Water too hot kills yeast, too cold won't activate it.\n\nFirst rise should happen in oiled bowl, covered, in warm spot (75-80°F) until doubled in size - usually 1-2 hours. Punch down gently and shape.\n\nSecond rise is shorter, 30-60 minutes until puffy but not necessarily doubled. Test with gentle finger poke - dough should spring back slowly when ready to bake.",
            "rejected": "Let dough sit in a warm place until it gets bigger.",
        },
        {
            "prompt": "What makes cookies chewy vs crispy?",
            "chosen": "Cookie texture depends on ingredient ratios, baking time, and temperature control.\n\nFor chewy cookies: Use more brown sugar than white (brown sugar's molasses adds moisture), slightly underbake, and use bread flour or add extra egg yolk for chewiness.\n\nFor crispy cookies: Use more white sugar, butter at room temperature, bake longer until edges are golden, and use all-purpose flour.\n\nBaking temperature also matters: lower temperature (325°F) spreads cookies more and creates chewier texture, higher temperature (375°F) sets edges quickly for crispier results.",
            "rejected": "Different ingredients and baking times make cookies different textures.",
        },
        {
            "prompt": "How do I keep salad greens fresh longer?",
            "chosen": "Proper storage dramatically extends salad green freshness.\n\nWash greens in cold water, then dry thoroughly using salad spinner or paper towels - excess moisture causes quick spoilage.\n\nStore in refrigerator in breathable container or plastic bag with paper towels to absorb moisture. Don't seal completely airtight - greens need some air circulation.\n\nFor delicate greens like arugula: use within 3-5 days. Hardier greens like romaine: 7-10 days. Remove any yellowing or slimy leaves immediately to prevent spread.",
            "rejected": "Keep salad in the fridge in a bag.",
        },
    ]
    return examples


def run_orpo_training(model, tokenizer):
    print("=" * 80)
    print("ODDS RATIO PREFERENCE OPTIMIZATION (ORPO)")
    print("=" * 80)
    print("Mathematical Foundation: L_ORPO = L_SFT + λ * L_OR")
    print(f"Training: {NUM_EPOCHS_ORPO} epochs, LR: {LEARNING_RATE_ORPO}")
    print("Innovation: Single-step SFT + preference optimization")

    cleanup_memory()

    orpo_data = create_orpo_dataset()
    dataset = Dataset.from_list(orpo_data)

    print(f"Dataset: {len(orpo_data)} preference pairs")
    print(f"Effective batch size: {BATCH_SIZE_ORPO * GRAD_ACCUM_ORPO}")
    print("Key advantage: No separate reference model needed (memory efficient)")

    try:
        from trl import ORPOTrainer, ORPOConfig

        config = ORPOConfig(
            output_dir="./temp_orpo",
            num_train_epochs=NUM_EPOCHS_ORPO,
            per_device_train_batch_size=BATCH_SIZE_ORPO,
            gradient_accumulation_steps=GRAD_ACCUM_ORPO,
            learning_rate=LEARNING_RATE_ORPO,
            max_length=MAX_LENGTH,
            max_prompt_length=MAX_LENGTH // 2,
            logging_steps=LOGGING_STEPS,
            save_strategy="no",
            fp16=False,
            bf16=torch.cuda.is_available(),
            warmup_ratio=WARMUP_RATIO,
            remove_unused_columns=False,
            gradient_checkpointing=True,
            dataloader_drop_last=True,
        )

        trainer = ORPOTrainer(
            model=model,
            args=config,
            train_dataset=dataset,
            processing_class=tokenizer,
        )

        print("\nStarting ORPO training...")
        print("Process: Monolithic SFT + preference learning in single phase")
        trainer.train()
        del trainer

    except Exception as e:
        print(f"ORPO training error: {e}")

    cleanup_memory()
    return model, tokenizer


def save_results_json(results, filename):
    os.makedirs("./results", exist_ok=True)
    with open(f"./results/{filename}", "w") as f:
        json.dump(
            {
                "timestamp": datetime.now().isoformat(),
                "model_outputs": results,
                "test_questions": TEST_QUESTIONS,
            },
            f,
            indent=2,
        )


def print_orpo_theory():
    """Print comprehensive ORPO theoretical foundation"""
    print("\n" + "=" * 80)
    print("ORPO THEORETICAL FOUNDATION")
    print("=" * 80)
    print("Mathematical Innovation: Monolithic Training Architecture")
    print()
    print("Loss Function:")
    print("  L_ORPO = L_SFT + λ * L_OR")
    print("  Where:")
    print("    L_SFT = Standard supervised fine-tuning loss")
    print("    L_OR = Odds ratio preference optimization loss")
    print("    λ = Weighting parameter for preference component")
    print()
    print("Odds Ratio Loss:")
    print("  L_OR = -log(σ(log(P(y_chosen|x) / P(y_rejected|x))))")
    print("  Where σ is the sigmoid function")
    print()
    print("Key Advantages:")
    print("  • No separate reference model needed (memory efficient)")
    print("  • Single training phase (time efficient)")
    print("  • Direct optimization of preferences during SFT")
    print("  • Unified objective function for both tasks")
    print()
    print("Training Process:")
    print("  1. Simultaneously optimize for task performance (SFT)")
    print("  2. Learn preference distinctions (OR)")
    print("  3. Balance both objectives with λ weighting")
    print("  4. Single model architecture handles both aspects")
    print()
    print("Computational Benefits:")
    print("  • Memory: ~50% reduction vs separate SFT+DPO")
    print("  • Time: Single training phase vs sequential")
    print("  • Complexity: Unified architecture vs multi-stage")
    print("  • Efficiency: Direct preference integration")
    print("=" * 80)


def main():
    print("=" * 80)
    print("ODDS RATIO PREFERENCE OPTIMIZATION (ORPO) - STANDALONE")
    print("=" * 80)
    print("Monolithic training: SFT + preference learning in single step")
    print("Innovation: Memory-efficient unified optimization")
    print("Mathematical foundation: Combined loss function approach")
    print("=" * 80)

    # Print theoretical foundation
    print_orpo_theory()

    install_packages()

    print(f"\nInitializing model: {MODEL_NAME}")

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )

    # Configure tokenizer
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.resize_token_embeddings(len(tokenizer))
    monitor_memory()

    # Evaluate base model
    print("\n" + "=" * 50)
    print("EVALUATING BASE MODEL")
    print("=" * 50)
    base_results = evaluate_stage(model, tokenizer, "BASE")
    save_results_json(base_results, "orpo_base_results.json")

    # Run ORPO training
    print("\n" + "=" * 50)
    print("STARTING ORPO TRAINING")
    print("=" * 50)
    model, tokenizer = run_orpo_training(model, tokenizer)

    # Evaluate trained model
    print("\n" + "=" * 50)
    print("EVALUATING TRAINED MODEL")
    print("=" * 50)
    trained_results = evaluate_stage(model, tokenizer, "ORPO")
    save_results_json(trained_results, "orpo_trained_results.json")

    # Save final model
    print(f"\nSaving ORPO-trained model...")
    os.makedirs("./models/orpo_standalone", exist_ok=True)
    model.save_pretrained("./models/orpo_standalone")
    tokenizer.save_pretrained("./models/orpo_standalone")

    # Compare results
    print(f"\n{'='*80}")
    print("COMPARATIVE ANALYSIS: Base vs ORPO")
    print(f"{'='*80}")
    print("Focus: Monolithic SFT + preference optimization benefits")
    print("Key aspects: Memory efficiency, unified training, quality improvement")
    print(f"{'='*80}")

    for i, question in enumerate(TEST_QUESTIONS, 1):
        print(f"\n[QUESTION {i}]: {question}")
        print("=" * 60)
        print(f"\n[BASE MODEL (No training)]:")
        print(f"{base_results[question]}")
        print(f"\n[ORPO MODEL (Monolithic SFT+preferences)]:")
        print(f"{trained_results[question]}")
        print("=" * 60)

    # Analysis summary
    print(f"\n{'='*80}")
    print("ORPO TRAINING ANALYSIS")
    print(f"{'='*80}")
    print("Technical Achievements:")
    print("  • Unified SFT and preference optimization in single phase")
    print("  • Eliminated need for separate reference model")
    print("  • Memory-efficient monolithic architecture")
    print("  • Direct preference integration during task learning")
    print()
    print("Efficiency Gains:")
    print("  • Memory: ~50% reduction vs traditional SFT→DPO pipeline")
    print("  • Training time: Single phase vs sequential training")
    print("  • Computational overhead: Unified objective function")
    print("  • Model complexity: Single architecture handles both tasks")
    print()
    print("Expected Improvements:")
    print("  • Enhanced response quality from preference integration")
    print("  • Better alignment with human quality standards")
    print("  • Improved task performance through unified optimization")
    print("  • More coherent preference-task learning integration")
    print()
    print("Mathematical Innovation:")
    print("  • L_ORPO = L_SFT + λ * L_OR combines both objectives")
    print("  • Odds ratio loss directly optimizes preference ranking")
    print("  • Single gradient update optimizes both components")
    print("  • Balanced weighting preserves both task and preference learning")
    print(f"{'='*80}")

    print(f"\n{'='*80}")
    print("ORPO TRAINING COMPLETED")
    print(f"{'='*80}")
    print("Results saved to:")
    print("  • ./models/orpo_standalone/ - Trained model")
    print("  • ./results/orpo_base_results.json - Base evaluation")
    print("  • ./results/orpo_trained_results.json - Trained evaluation")
    print()
    print("Key Innovation:")
    print("  ORPO demonstrates that SFT and preference optimization")
    print("  can be effectively combined in a single, memory-efficient training phase")
    print(f"{'='*80}")

In [2]:
# Execute main
if __name__ == "__main__":
    main()

ODDS RATIO PREFERENCE OPTIMIZATION (ORPO) - STANDALONE
Monolithic training: SFT + preference learning in single step
Innovation: Memory-efficient unified optimization
Mathematical foundation: Combined loss function approach

ORPO THEORETICAL FOUNDATION
Mathematical Innovation: Monolithic Training Architecture

Loss Function:
  L_ORPO = L_SFT + λ * L_OR
  Where:
    L_SFT = Standard supervised fine-tuning loss
    L_OR = Odds ratio preference optimization loss
    λ = Weighting parameter for preference component

Odds Ratio Loss:
  L_OR = -log(σ(log(P(y_chosen|x) / P(y_rejected|x))))
  Where σ is the sigmoid function

Key Advantages:
  • No separate reference model needed (memory efficient)
  • Single training phase (time efficient)
  • Direct optimization of preferences during SFT
  • Unified objective function for both tasks

Training Process:
  1. Simultaneously optimize for task performance (SFT)
  2. Learn preference distinctions (OR)
  3. Balance both objectives with λ weighting
 

`torch_dtype` is deprecated! Use `dtype` instead!


GPU Memory: 0.92GB allocated, 1.19GB reserved

EVALUATING BASE MODEL

BASE MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: How long should it be cooked?
What is the best way to cook spaghetti?
Which method of cooking is the best for spaghetti?
Do those questions have the same meaning?
Options:
a). no.
b). yes. a). No.

The two questions you asked are asking about different things and have slightly different meanings. Let's break down each question:

1. "How do I cook pasta perfectly? How long should it be cooked?"
   - This question is asking about how to cook pasta in a specific time frame. It implies that you want to know the optimal cooking time, temperature, or consistency.

2. "What is the best way to cook spaghetti?"
   - This question is asking for the most effective methods for making spaghetti. It could mean different things depending on context:
     - If it refers to traditional Italian cooking methods (such as slow-cooki

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting ORPO training...
Process: Monolithic SFT + preference learning in single phase


Step,Training Loss
5,2.237600
10,0.485800
15,0.037400



EVALUATING TRAINED MODEL

ORPO MODEL EVALUATION

Q1: How do I cook pasta perfectly?
----------------------------------------
Answer: Perfect pasta requires proper salt and moisture level, as well as precise timing.

Preheat oven to 70-80°C (150-175°F). In large bowl, combine warm water and salt. Add pasta and toss to break up any clumps.

Cook 4-5 minutes until fork-tender. This should take 7-10 minutes depending on type and thickness of pasta. Test with plastic wrap - undercooked pasta has a mushy texture.

Q2: What's the best way to scramble eggs?
----------------------------------------
Answer: Perfect scrambled eggs require just the right temperature and time.

Heat cast iron pan or non-stick cooking surface over high heat. Add small amount of oil first, without splattering. Scan kitchen and add extra egg yolk while hot - this ensures even scrambling.

Add first 3-4 eggs, breaking up any clumps with your hands if needed. Cook until done, about 5-7 minutes. Drain thoroughly and let